In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# load the predictions_patient_data.csv
patient_data = pd.read_csv('Data/patient_data_final.csv')

In [ ]:
# print column names
patient_data.columns

In [ ]:
# print columns and their data types
patient_data.dtypes

## Exploratory Data Analysis

In [ ]:
patient_data.shape

In [ ]:
# Encode the actual_label column
label_encoder = LabelEncoder()
patient_data['actual_label'] = label_encoder.fit_transform(patient_data['actual_label'])

In [ ]:
# plot correlation matrix

plt.figure(figsize=(12, 8))
sns.heatmap(patient_data.corr(), annot=True)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# remove unique_patient_ID and prediction_label columns
patient_data = patient_data.drop(['unique_patient_ID', 'predicted_label'], axis=1)

# remove rows where actual_label is 'Unknown'
patient_data = patient_data[patient_data['actual_label'] != 'Unknown']

In [ ]:
patient_data.shape

In [ ]:
patient_data.head()

In [ ]:
# plot actual_label distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='actual_label', data=patient_data)
plt.title('Actual Label Distribution')
plt.show()

## Random Forest Classifier

In [ ]:
# Split the dataset into features and target
X = patient_data.drop('actual_label', axis=1)  # Features
y = patient_data['actual_label']  # Target

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numeric columns
numeric_cols = ['M', 'age_at_initial_pathologic_diagnosis', 'lymphnodesremoved', 
                'lymphnodesinvaded', 'stageall']

# Initialize the scaler
scaler = MinMaxScaler()

# Scale the numeric columns
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# One-hot encoding
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

In [ ]:
# plot actual_label distribution
plt.figure(figsize=(10, 6))
sns.countplot(x=y_train)
plt.title('Actual Label Distribution')
plt.show()

In [ ]:
# get percentage distribution of actual_label in total data
import matplotlib.pyplot as plt

# Assuming the distribution of 'actual_label' as example values
labels = ['MSS', 'MSI']
values = [70, 30]  # Example percentages for MSS and MSI

# Define custom color for the plot using normalized RGB
custom_color = (21/255, 96/255, 130/255)  # Normalized RGB for matplotlib

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color=custom_color)
plt.title('Distribution of Patient Labels in the Dataset', fontsize=14, color='black')
plt.xlabel('', fontsize=12, color='black')
plt.ylabel('Percentage (%)', fontsize=12, color='black')
plt.xticks(color='black')
plt.yticks(color='black')
plt.tight_layout()
plt.show()


In [ ]:
# Initialize the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=50, random_state=42)

# Fit the model using X_train prepared with get_dummies
# X_train = pd.get_dummies(X_train)
rf_model.fit(X_train, y_train)

# reindex X_test to have the same columns as X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)  # Add columns with default value 0

In [ ]:
X_train.head()

In [ ]:
# Make predictions on the test data
rf_predictions = rf_model.predict(X_test)

In [ ]:
# print first few actual and predicted labels
print('Actual   :', y_test.values[0:20])
print('Predicted:', rf_predictions[0:20])

In [ ]:
# Calculate the accuracy of the model using cross-validation
from sklearn.model_selection import cross_val_score

# get test set accuracy
accuracy = accuracy_score(y_test, rf_predictions)
print('Accuracy:', accuracy)

In [ ]:
# classification report for random forest model
print(classification_report(y_test, rf_predictions, target_names=label_encoder.classes_))

In [ ]:
# print total rows in train and test data
print('Train data:', X_train.shape)
print('Test data:', X_test.shape)

In [ ]:
# import seaborn and confusion matrix
from sklearn.metrics import confusion_matrix

# plot confusion matrix
plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix(y_test, rf_predictions), annot=True, fmt='d', cmap='Reds')
plt.title('Confusion Matrix')
plt.show()

## Addtional results with best model (Random Forest)

In [ ]:
# plot how age_at_initial_pathologic_diagnosis affects the actual_label
plt.figure(figsize=(10, 6))
sns.boxplot(x='actual_label', y='age_at_initial_pathologic_diagnosis', data=patient_data)
plt.title('Age at Initial Pathologic Diagnosis vs Actual Label')
plt.show()

In [ ]:
# Feature Importance

# Assuming 'rf_model.feature_importances_' and 'X_train.columns' are defined:
# Extract feature importances from the model
feature_importance = rf_model.feature_importances_

# Create a DataFrame to hold feature names and their corresponding importance
# Rename 'predicted_probability' to 'predicted_probability_from_patho_image' where applicable
feature_importance_df = pd.DataFrame({
    'Feature': ['predicted_probability_from_patho_image' if x == 'predicted_probability' else x for x in X_train.columns],
    'Importance': feature_importance
})

# Sort the DataFrame to display the most important features at the top
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Define custom color for the plot using normalized RGB
custom_color = (21/255, 96/255, 130/255)  # Normalized RGB for matplotlib

# Plotting the feature importance
plt.figure(figsize=(10, 6))  # Set the figure size for better readability
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10), color=custom_color)
plt.title('Top 10 Most Significant Features in CRC Diagnosis', fontsize=14, color='black')
plt.xlabel('Importance', fontsize=12, color='black')
plt.ylabel('', fontsize=12, color='black')
plt.xticks(color='black')
plt.yticks(color='black')
plt.tight_layout()  # Adjust the layout to make sure everything fits without overlap
plt.show()  # Display the plot

In [ ]:
# sensitivity and specificity

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_test, rf_predictions)

# Calculate the sensitivity
sensitivity = conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])

# Calculate the specificity
specificity = conf_matrix[1, 1] / (conf_matrix[1, 0] + conf_matrix[1, 1])

print('Sensitivity:', sensitivity)
print('Specificity:', specificity)

In [ ]:
# create function to calculate Matthew’s correlation coefficient

def calculate_mcc(y_true, y_pred):
    conf_matrix = confusion_matrix(y_true, y_pred)
    tp = conf_matrix[0, 0]
    tn = conf_matrix[1, 1]
    fp = conf_matrix[0, 1]
    fn = conf_matrix[1, 0]
    mcc = (tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return mcc

In [ ]:
# mathews correlation coefficient

# Calculate the confusion matrix

conf_matrix = confusion_matrix(y_test, rf_predictions)

# Calculate the MCC using the function we created
mcc = calculate_mcc(y_test, rf_predictions)

print('MCC:', mcc)

In [ ]:
# roc-auc score

from sklearn.metrics import roc_auc_score

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(y_test, rf_predictions)

print('ROC-AUC:', roc_auc)

In [ ]:
# plot distribution of actual_label in the dataset

plt.figure(figsize=(10, 6))  # Set the figure size for better readability

# Custom color as defined earlier
custom_color = (21/255, 96/255, 130/255)

# Create a mapping of the actual labels to more descriptive names
label_names = {0: 'MSI', 1: 'MSS'}
patient_data['label_name'] = patient_data['actual_label'].map(label_names)  # Map numeric labels to names

# Plotting the actual label distribution with named labels
ax = sns.countplot(x='label_name', data=patient_data, color=custom_color)
plt.title('Class Distribution In The Patient Dataset', fontsize=14, color='black')  # Match title style
plt.xlabel('', fontsize=12, color='black')  # Set and label x-axis
plt.ylabel('Count', fontsize=12, color='black')  # Set and label y-axis
plt.xticks(color='black')  # Match tick color
plt.yticks(color='black')

plt.show()  # Display the plot



In [ ]:
# get number for each class in actual_label for total data
class_distribution = patient_data['actual_label'].value_counts()
print(class_distribution)

In [ ]:
# get number of each class in actual_label for train data
train_class_distribution = y_train.value_counts()
print(train_class_distribution)

# get number of each class in actual_label for test data
test_class_distribution = y_test.value_counts()
print(test_class_distribution)